# Treino do aluno (SFT + LoRA) na T4 do Colab

Fluxo: **dataset exportado pela API → treino → merge → GGUF → download → `ollama create` → `POST /treinos`**.

Antes de rodar: `Ambiente de execução → Alterar tipo → GPU (T4)`. Tenha em mãos `treino.jsonl`, `held_out.jsonl` e a `versao` do manifesto (`GET /api/v1/datasets`).

Tempo esperado (Qwen3-1.7B, ~3.000 exemplos, 2 épocas): 1 a 2 h. O held-out entra **só** no eval loss; o script recusa rodar se algum id estiver nos dois arquivos.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Código do projeto

Clone o repositório (ajuste a URL) **ou** faça upload da pasta `treino/` pelo painel de arquivos.

In [ ]:
import os
REPO_URL = "https://github.com/SEU_USUARIO/GENA_TCC_UFG.git"  # ajuste
if not os.path.exists("repo"):
    !git clone --depth 1 $REPO_URL repo
%cd repo
!pip install -q -r treino/requirements-treino.txt

## 2. Dataset

Faça upload de `treino.jsonl` e `held_out.jsonl` (baixados de `GET /api/v1/datasets/{versao}/arquivo/{split}`) e informe a versão.

In [ ]:
from google.colab import files
DATASET_VERSAO = "COLE_A_VERSAO_DO_MANIFESTO"  # ex.: ab12cd34ef56
enviados = files.upload()
assert "treino.jsonl" in enviados, "envie treino.jsonl"
print({nome: len(conteudo) for nome, conteudo in enviados.items()})

## 3. Dry-run (valida os dados e o plano, sem gastar GPU)

In [ ]:
HELD_OUT = "--held-out held_out.jsonl" if os.path.exists("held_out.jsonl") else ""
!python -m treino.sft_lora --perfil colab-t4 --dataset treino.jsonl $HELD_OUT --dataset-versao $DATASET_VERSAO --saida saida --dry-run

## 4. Treino

`--perfil colab-t4` = Qwen3-1.7B, fp16, LoRA r=16, seq 1024, 2 épocas. Para o 4B em 4-bit use `colab-t4-4bit` (sem merge automático). Para a condição "sem raciocínio", exporte o dataset com `incluir_raciocinio=false` na API; o script não muda.

In [ ]:
!python -m treino.sft_lora --perfil colab-t4 --dataset treino.jsonl $HELD_OUT --dataset-versao $DATASET_VERSAO --saida saida

In [ ]:
import json
run = json.load(open("saida/run.json", encoding="utf-8"))
print(json.dumps({k: run[k] for k in ("status", "modelo_base", "duracao_s", "metricas", "modelo_local_id", "hardware")}, indent=2, ensure_ascii=False))

## 5. GGUF quantizado (o que o Ollama carrega em CPU)

In [ ]:
!bash treino/exportar_gguf.sh saida/merged saida/gguf

## 6. Download

Baixe os três arquivos (ou copie para o Drive com `drive.mount`). Na máquina do Ollama, na pasta com os arquivos:

```
ollama create <modelo_local_id sem o prefixo local:> -f Modelfile
```

Depois registre o run na API com o `run.json` como corpo (`artefato` pode apontar para o caminho do GGUF na sua máquina):

```
curl -X POST -H "X-API-Key: SUA_CHAVE" -H "Content-Type: application/json" --data-binary @run.json http://localhost:8000/api/v1/treinos
curl -X POST -H "X-API-Key: SUA_CHAVE" http://localhost:8000/api/v1/treinos/<id>/verificar
```

In [ ]:
files.download("saida/gguf/modelo-q4_k_m.gguf")
files.download("saida/gguf/Modelfile")
files.download("saida/run.json")